<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D1%81%D0%B2%D0%B5%D0%B6_%D0%A1%D0%B0%D0%B1_%D0%9F%D0%BE%D1%85_%D1%8E%D0%B7%D0%B5%D1%80%D1%8B_%D0%91%D0%B5%D0%B7_%D0%92%D0%B7%D0%B0%D0%B8%D0%BC_%D0%9E%D0%B3%D1%80_100_ml_ozon_recsys_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OZON RecSys Baseline - Рекомендательная система для категории Apparel
Этот ноутбук содержит базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.

## Задача
- Предсказать топ-100 товаров для каждого пользователя из тестовой выборки
- Метрика оценки: NDCG@100
- Данные: ~38GB в формате parquet, 1.6B взаимодействий, 19M заказов


In [1]:
# === 1. Монтируем Google Drive, задаём пути к данным (структура Colab/Яндекс) ===
from google.colab import drive
drive.mount('/content/drive')

# Шаг 1. Импорты и пути (Colab/локально, без лишних библиотек)
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict, Counter, deque # Добавлены для подсчета популярности
import glob

# Пути к данным (замените на свои)
ORDERS_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_orders_data/final_apparel_orders_data_07'
ORDERS_PATH2 = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/raw/ml_ozon_recsys_train_final_apparel_orders_data'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/ml_ozon_recsys_test'

# Шаг 2. Загрузка всех заказов (train)
def load_orders():
    print("Загружаем тренировочные данные заказов...")
    orders = []
    for path in [ORDERS_PATH, ORDERS_PATH2]:
        for f in Path(path).rglob('*.parquet'):
            orders.append(pd.read_parquet(f))
    df = pd.concat(orders, ignore_index=True)
    # Обогащаем created_date, если нужно
    if 'created_date' in df.columns and df['created_date'].isna().sum():
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_date'].fillna(df['created_timestamp'].dt.date)
        df['created_date'] = pd.to_datetime(df['created_date'])
    elif 'created_date' not in df.columns and 'created_timestamp' in df.columns:
        # Если created_date вообще отсутствует, создаем её
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_timestamp'].dt.date
        df['created_date'] = pd.to_datetime(df['created_date'])
    print(f"Загружено заказов: {len(df):,}")
    return df

orders_df = load_orders()

Mounted at /content/drive
Загружаем тренировочные данные заказов...
Загружено заказов: 20,362,338


In [2]:
print("=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(orders_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in orders_df.columns:
    print(f"  • {col}: {orders_df[col].dtype}")

=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|    |   item_id |   user_id | created_timestamp          | last_status      | last_status_timestamp   | created_date        |
+====+===========+===========+============================+==================+=========================+=====================+
|  0 | 332361399 |      2841 | 2025-07-14 08:38:22.250000 | proccesed_orders | 2025-07-14 11:35:22     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|  1 | 153141296 |      3681 | 2025-07-14 09:13:56.410000 | proccesed_orders | 2025-07-14 10:47:53     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+

📊 Схема данных:
  • ite

In [3]:
# Шаг 3. Загрузка тестовых пользователей
def load_test_users():
    print("Загружаем тестовых пользователей...")
    test_files = glob.glob(f'{TEST_PATH}/*.parquet')
    users = set()
    for f in tqdm(test_files, desc="Обработка тестовых файлов"):
        df_part = pd.read_parquet(f)
        if 'user_id' in df_part.columns:
            users.update(df_part['user_id'].unique())
    print(f"Найдено уникальных тестовых пользователей: {len(users):,}")
    return list(users)

test_users = load_test_users()

Загружаем тестовых пользователей...


Обработка тестовых файлов: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]

Найдено уникальных тестовых пользователей: 470,347


In [4]:
print("=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
test_users_df = pd.DataFrame({'user_id': test_users})
print(f"📊 Размер: {len(test_users_df):,} строк")
print("\n👉 Первые 2 строки:")
print(test_users_df.head(2).to_string())
print("\n📋 Колонки и типы данных:")
for col in test_users_df.columns:
    print(f"  • {col:20} {test_users_df[col].dtype}")

=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
📊 Размер: 470,347 строк

👉 Первые 2 строки:
   user_id
0        1
1  3145730

📋 Колонки и типы данных:
  • user_id              int32


In [6]:
print("\n\nАНАЛИЗ ЗАКАЗОВ")
print("=" * 50)
print(f"Общее количество заказов: {len(orders_df):,}")
print(f"Уникальных пользователей: {orders_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {orders_df['item_id'].nunique():,}")

if 'created_date' in orders_df.columns:
    min_date = orders_df['created_date'].min().date()
    max_date = orders_df['created_date'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'last_status' in orders_df.columns:
    print("\nРаспределение статусов заказов:")
    status_counts = orders_df['last_status'].value_counts()
    status_counts_pct = orders_df['last_status'].value_counts(normalize=True) * 100
    for status, count in status_counts.items():
        pct = status_counts_pct[status]
        print(f"  {status}: {count:,} ({pct:.1f}%)")




АНАЛИЗ ЗАКАЗОВ
Общее количество заказов: 20,362,338
Уникальных пользователей: 842,254
Уникальных товаров: 4,679,218
Период данных: 2025-01-01 - 2025-07-15

Распределение статусов заказов:
  delivered_orders: 10,420,894 (51.2%)
  canceled_orders: 8,420,631 (41.4%)
  proccesed_orders: 1,520,813 (7.5%)


In [5]:
print("\nТоп-10 самых популярных товаров (по количеству заказов):")
top_items_orders = orders_df['item_id'].value_counts().head(10)
for item_id, count in top_items_orders.items():
    print(f"  Товар {item_id}: {count:,} заказов")



Топ-10 самых популярных товаров (по количеству заказов):
  Товар 51974017: 13,361 заказов
  Товар 187052809: 12,384 заказов
  Товар 207631139: 8,877 заказов
  Товар 143497612: 4,096 заказов
  Товар 119105606: 3,497 заказов
  Товар 247423473: 3,188 заказов
  Товар 77696741: 2,735 заказов
  Товар 175287070: 2,725 заказов
  Товар 285009143: 2,634 заказов
  Товар 201930716: 2,624 заказов


In [6]:
# Шаг 4. Аналитическая "модель популярности" по всем покупкам за последние 2 недели

def analytic_popularity_order(orders_df, top_k=100, start_date='2025-07-02', end_date='2025-07-15'):
    # Только доставленные заказы за финальный период
    orders = orders_df[
        (orders_df['last_status']=='delivered_orders') &
        (orders_df['created_date'] >= pd.to_datetime(start_date)) &
        (orders_df['created_date'] <= pd.to_datetime(end_date))
    ]
    # Формируем user_preferences: порядок покупок
    orders = orders.sort_values(['user_id', 'created_timestamp'])
    user_prefs = {}
    for uid, group in orders.groupby('user_id'):
        items = group['item_id'].tolist()
        user_prefs[str(uid)] = items
    # Выбираем ТОП-K товаров по частоте (можно без сортировки по позициям)
    top_items = orders['item_id'].value_counts().head(top_k).index.tolist()
    return top_items

popular_items = analytic_popularity_order(orders_df, top_k=100)

In [7]:
# Выводим топ-10 популярных товаров
print("Топ-10 популярных товаров:")
for i, item_id in enumerate(popular_items[:10], 1):
    print(f"{i:2d}. {item_id}")

Топ-10 популярных товаров:
 1. 51974017
 2. 175287070
 3. 166327353
 4. 187052809
 5. 247423473
 6. 11083343
 7. 334086992
 8. 63987378
 9. 206494927
10. 201930716


In [8]:
# Шаг 6. Получение тестовых пользователей
test_users_during_period = list(set(test_users))
print(f"\n=== 👥 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
print(f"Найдено тестовых пользователей: {len(test_users_during_period):,}")


=== 👥 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
Найдено тестовых пользователей: 470,347


In [9]:
# Исправленный Шаг 7. Вычисление истории покупок (для ВСЕХ пользователей в orders_df с delivered статусом)
def build_user_history_full_period(orders_df):
    """
    Построить историю покупок для ВСЕХ пользователей в orders_df.
    Это необходимо, чтобы искать похожих среди всех, кто имеет историю.
    """
    print("\n=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (для ВСЕХ пользователей с delivered_orders) ===")

    # Все доставленные заказы - история ВСЕХ пользователей с такими заказами
    # Убираем фильтр по test_users
    user_history = orders_df[
        (orders_df['last_status'] == 'delivered_orders')
    ].groupby('user_id')['item_id'].apply(set).to_dict()

    # Конвертируем в список для совместимости
    user_preferences = {uid: list(items) for uid, items in user_history.items()}

    print(f"История покупок построена для {len(user_preferences):,} пользователей (все с delivered)")
    return user_preferences

# --- Вызов исправленной функции ---
# Теперь user_preferences будет содержать историю для всех пользователей из orders_df со статусом delivered
user_preferences = build_user_history_full_period(orders_df)



=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (для ВСЕХ пользователей с delivered_orders) ===
История покупок построена для 805,981 пользователей (все с delivered)


In [20]:
def build_similar_users_dict_superfast(user_preferences, test_users, min_common_items=2, top_similar=5):
    """
    Супербыстрая версия: использует обратный индекс товаров для поиска похожих пользователей.

    Args:
        user_preferences: словарь {user_id: [список купленных товаров]}
        test_users: список ID тестовых пользователей
        min_common_items: минимальное количество общих товаров для считания пользователей похожими
        top_similar: максимальное количество похожих пользователей для хранения

    Returns:
        similar_users_dict: словарь {user_id: [(similar_user_id, similarity_score), ...]}
    """
    print("\n=== 👥 ПОСТРОЕНИЕ СЛОВАРЯ ПОХОЖИХ ПОЛЬЗОВАТЕЛЕЙ (СУПЕР-ОПТИМИЗИРОВАНО) ===")

    # Инициализация словаря похожих пользователей
    similar_users_dict = {uid: [] for uid in test_users}

    # Преобразуем списки покупок в множества для быстрого вычисления пересечений
    user_items_sets = {uid: set(items) for uid, items in user_preferences.items()}

    # Создаем обратный индекс: для каждого товара список пользователей, купивших его
    item_to_users = defaultdict(set)
    for uid, items in user_preferences.items():
        for item in items:
            item_to_users[item].add(uid)

    # Для каждого тестового пользователя находим похожих
    for user_id in tqdm(test_users, desc="Поиск похожих пользователей"):
        # Если пользователя нет в истории покупок, пропускаем
        if user_id not in user_items_sets:
            continue

        user_items = user_items_sets[user_id]

        # Если у пользователя нет покупок, пропускаем
        if not user_items:
            continue

        # Для подсчета общих товаров между пользователями
        common_items_count = Counter()

        # Для каждого товара текущего пользователя
        for item in user_items:
            # Находим других пользователей, купивших этот товар
            for other_id in item_to_users[item]:
                if other_id != user_id:
                    common_items_count[other_id] += 1

        # Отбираем пользователей с достаточным количеством общих товаров
        candidates = [other_id for other_id, count in common_items_count.items()
                     if count >= min_common_items]

        similarities = []
        for other_id in candidates:
            other_items = user_items_sets[other_id]
            # Жаккард: |A ∩ B| / |A ∪ B|
            similarity = common_items_count[other_id] / len(user_items.union(other_items))
            similarities.append((other_id, similarity))

        # Сортируем по убыванию схожести и берем top_similar
        similarities.sort(key=lambda x: x[1], reverse=True)
        similar_users_dict[user_id] = similarities[:top_similar]

    print(f"Словарь похожих пользователей построен для {len(similar_users_dict)} тестовых пользователей")

    return similar_users_dict


In [ ]:
# Вызов функции
similar_users_dict = build_similar_users_dict_superfast(
    user_preferences=user_preferences,
    test_users=test_users_during_period,
    min_common_items=1,  # Минимальное количество общих товаров
    top_similar=5  # Максимальное количество похожих пользователей для хранения
)



=== 👥 ПОСТРОЕНИЕ СЛОВАРЯ ПОХОЖИХ ПОЛЬЗОВАТЕЛЕЙ (СУПЕР-ОПТИМИЗИРОВАНО) ===


Поиск похожих пользователей:   5%|▍         | 22099/470347 [02:42<36:11, 206.39it/s]

In [25]:
# === Статистика по словарю похожих пользователей ===
print("\n📊 СТАТИСТИКА СЛОВАРЯ ПОХОЖИХ ПОЛЬЗОВАТЕЛЕЙ")
print("=" * 50)

# Базовая статистика
similarity_counts = [len(similar) for similar in similar_users_dict.values()]
print(f"Всего пользователей в словаре: {len(similar_users_dict):,}")

print(f"\n📈 Распределение по количеству похожих:")
print(f"  Среднее количество похожих пользователей: {np.mean(similarity_counts):.2f}")
print(f"  Медиана: {np.median(similarity_counts):.1f}")
print(f"  Максимальное количество: {max(similarity_counts)}")
print(f"  Минимальное количество: {min(similarity_counts)}")

# Подсчет пользователей по категориям
zero_similar = sum(1 for x in similarity_counts if x == 0)
one_similar = sum(1 for x in similarity_counts if x == 1)
two_similar = sum(1 for x in similarity_counts if x == 2)
three_similar = sum(1 for x in similarity_counts if x == 3)
four_similar = sum(1 for x in similarity_counts if x == 4)

print(f"\n📋 Детальное распределение:")
print(f"  Без похожих: {zero_similar:,} ({zero_similar/len(similarity_counts)*100:.1f}%)")
print(f"  1 похожий:   {one_similar:,} ({one_similar/len(similarity_counts)*100:.1f}%)")
print(f"  2 похожих:   {two_similar:,} ({two_similar/len(similarity_counts)*100:.1f}%)")
print(f"  3 похожих:   {three_similar:,} ({three_similar/len(similarity_counts)*100:.1f}%)")
print(f"  4 похожих:   {four_similar:,} ({four_similar/len(similarity_counts)*100:.1f}%)")

# Статистика по схожести
all_similarities = []
for user_similar in similar_users_dict.values():
    for _, similarity in user_similar:
        all_similarities.append(similarity)

if all_similarities:
    print(f"\n📊 Статистика значений схожести:")
    print(f"  Средняя схожесть: {np.mean(all_similarities):.4f}")
    print(f"  Медианная схожесть: {np.median(all_similarities):.4f}")
    print(f"  Минимальная схожесть: {np.min(all_similarities):.4f}")
    print(f"  Максимальная схожесть: {np.max(all_similarities):.4f}")

# Пример структуры данных
print(f"\n🔍 Пример структуры словаря:")
sample_user_id = list(similar_users_dict.keys())[0] if similar_users_dict else None
if sample_user_id:
    sample_data = similar_users_dict[sample_user_id]
    print(f"  Пользователь {sample_user_id}:")
    print(f"    Тип: {type(sample_data)}")
    print(f"    Количество: {len(sample_data)}")
    if sample_data:
        print(f"    Первый элемент: {sample_data[0]}")
        print(f"    Тип элемента: {type(sample_data[0])}")

print("\n" + "=" * 50)


📊 СТАТИСТИКА СЛОВАРЯ ПОХОЖИХ ПОЛЬЗОВАТЕЛЕЙ
Всего пользователей в словаре: 470,347

📈 Распределение по количеству похожих:
  Среднее количество похожих пользователей: 4.66
  Медиана: 5.0
  Максимальное количество: 5
  Минимальное количество: 0

📋 Детальное распределение:
  Без похожих: 25,512 (5.4%)
  1 похожий:   3,475 (0.7%)
  2 похожих:   2,863 (0.6%)
  3 похожих:   2,744 (0.6%)
  4 похожих:   2,345 (0.5%)

📊 Статистика значений схожести:
  Средняя схожесть: 0.1070
  Медианная схожесть: 0.0714
  Минимальная схожесть: 0.0009
  Максимальная схожесть: 1.0000

🔍 Пример структуры словаря:
  Пользователь 1:
    Тип: <class 'list'>
    Количество: 5
    Первый элемент: (3402181, 0.07142857142857142)
    Тип элемента: <class 'tuple'>



In [24]:
import pickle # <-- Добавьте эту строку
from pathlib import Path

# === 5. Сохранение данных ===
# Создаем путь для сохранения
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed'
Path(SAVE_PATH).mkdir(parents=True, exist_ok=True)

print("=== 💾 СОХРАНЕНИЕ ДАННЫХ ДЛЯ БЫСТРОЙ ЗАГРУЗКИ ===")

# Сохраняем словарь похожих пользователей
similar_users_file = f"{SAVE_PATH}/items1_similar_users_dict.pkl"
with open(similar_users_file, 'wb') as f:
    pickle.dump(similar_users_dict, f)
print(f"✅ Похожие пользователи сохранены: {similar_users_file}")

=== 💾 СОХРАНЕНИЕ ДАННЫХ ДЛЯ БЫСТРОЙ ЗАГРУЗКИ ===
✅ Похожие пользователи сохранены: /content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed/items1_similar_users_dict.pkl


In [26]:
import pickle

# === 📥 ЗАГРУЗКА СОХРАНЕННЫХ ДАННЫХ ===
print("=== 📥 ЗАГРУЗКА СОХРАНЕННЫХ ДАННЫХ ===")

# Путь к сохраненному файлу (должен совпадать с путем сохранения)
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed'
similar_users_file = f"{SAVE_PATH}/items1_similar_users_dict.pkl"

# Загрузка словаря похожих пользователей (используем исходное имя переменной)
try:
    with open(similar_users_file, 'rb') as f:
        similar_users_dict = pickle.load(f) # <-- Имя переменной такое же
    print(f"✅ Словарь похожих пользователей загружен: {similar_users_file}")
    print(f"Количество пользователей в загруженном словаре: {len(similar_users_dict):,}")

    # Проверка: выведем несколько записей
    print("\n🔍 Примеры загруженных данных:")
    sample_keys = list(similar_users_dict.keys())[:3] # Первые 3 ключа
    for key in sample_keys:
        print(f"  Пользователь {key}: {similar_users_dict[key]}")

except FileNotFoundError:
    print(f"❌ Файл {similar_users_file} не найден. Убедитесь, что путь корректен и файл был сохранен.")
    similar_users_dict = {} # Инициализируем пустым словарем

except Exception as e:
    print(f"❌ Ошибка при загрузке файла: {e}")
    similar_users_dict = {} # Инициализируем пустым словарем

=== 📥 ЗАГРУЗКА СОХРАНЕННЫХ ДАННЫХ ===
✅ Словарь похожих пользователей загружен: /content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed/items1_similar_users_dict.pkl
Количество пользователей в загруженном словаре: 470,347

🔍 Примеры загруженных данных:
  Пользователь 1: [(3402181, 0.07142857142857142), (2996180, 0.05405405405405406), (2202781, 0.05), (1645140, 0.05), (1979990, 0.05)]
  Пользователь 3145730: []
  Пользователь 3145731: [(71721, 0.16666666666666666), (4365911, 0.16666666666666666), (339591, 0.16666666666666666), (3862801, 0.16666666666666666), (1928660, 0.16666666666666666)]


In [27]:
# Шаг 7. Вычисление истории покупок для тестовых пользователей (их покупки во всем периоде)
def build_user_history_full_period(orders_df, test_users):
    """Построить историю покупок тестовых пользователей во всем периоде"""
    print("\n=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (весь период) ===")

    # Все доставленные заказы тестовых пользователей во всем периоде
    user_history = orders_df[
        (orders_df['last_status'] == 'delivered_orders') &
        (orders_df['user_id'].isin(test_users))
    ].groupby('user_id')['item_id'].apply(set).to_dict()

    # Конвертируем в список для совместимости
    user_preferences = {uid: list(items) for uid, items in user_history.items()}

    print(f"История покупок построена для {len(user_preferences):,} пользователей")
    return user_preferences

user_preferences_test = build_user_history_full_period(orders_df, test_users_during_period)


=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (весь период) ===
История покупок построена для 449,231 пользователей


In [28]:
# Шаг 8. Генерация рекомендаций
def generate_recommendations(test_users, popular_items, user_preferences, top_k=100):
    """Генерация рекомендаций: популярные товары, исключая уже купленные"""
    print("\n=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ ===")
    recs = {}
    for uid in tqdm(test_users, desc='Генерация рекомендаций'):
        bought = set(user_preferences.get(uid, []))
        recs[uid] = [item for item in popular_items if item not in bought][:top_k]
    print(f"Рекомендации сгенерированы для {len(recs):,} пользователей")
    return recs

recommendations = generate_recommendations(test_users_during_period, popular_items, user_preferences_test)


=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ ===


Генерация рекомендаций: 100%|██████████| 470347/470347 [00:08<00:00, 57536.52it/s] 


Рекомендации сгенерированы для 470,347 пользователей


In [29]:
def generate_recommendations_hybrid(
    test_users,
    popular_items,
    user_preferences,
    similar_users_dict,
    orders_df,
    top_k=100,
    start_date='2025-07-02',  # Тот же период, что и для популярных товаров
    end_date='2025-07-15'     # Тот же период, что и для популярных товаров
):
    """
    Генерация гибридных рекомендаций:
    - Для пользователей с историей и похожими: CF на базе пользователей.
    - Для остальных: популярные товары, исключая уже купленные.
    """
    print("\n=== 🎯 ГЕНЕРАЦИЯ ГИБРИДНЫХ РЕКОМЕНДАЦИЙ ===")

    # 1. Подготовка: создадим словарь покупок за последние 2 недели для быстрого доступа
    print("Подготовка индекса покупок пользователей за последние 2 недели...")

    # Фильтруем заказы по тому же периоду, что и для популярных товаров
    recent_orders = orders_df[
        (orders_df['last_status'] == 'delivered_orders') &
        (orders_df['created_date'] >= pd.to_datetime(start_date)) &
        (orders_df['created_date'] <= pd.to_datetime(end_date))
    ]

    # Создаем индекс только из недавних покупок
    user_items_index = recent_orders.groupby('user_id')['item_id'].apply(set).to_dict()

    print(f"Индекс покупок создан для {len(user_items_index)} пользователей за период {start_date} - {end_date}.")

    recommendations = {}

    for uid in tqdm(test_users, desc='Генерация рекомендаций'):
        rec_items = []

        # Получаем историю текущего пользователя (то, что он уже купил)
        bought_by_user = set(user_preferences.get(uid, []))

        # Получаем список похожих пользователей
        similar_users_list = similar_users_dict.get(uid, [])

        # --- Логика гибридной модели ---
        if uid in user_preferences and similar_users_list:
            # --- Сценарий 1: Есть история и есть похожие пользователи ---
            # Используем Collaborative Filtering

            item_scores = defaultdict(float)

            # Проходим по каждому похожему пользователю
            for similar_user_id, similarity_score in similar_users_list:
                # Получаем НЕДАВНИЕ товары, купленные похожим пользователем
                items_bought_by_similar = user_items_index.get(similar_user_id, set())

                # Добавляем эти товары в счетчик, взвешивая по степени схожести
                for item in items_bought_by_similar:
                    # Убедимся, что товар еще не куплен целевым пользователем
                    if item not in bought_by_user:
                         # Увеличиваем "счет" товара на "силу" схожести похожего пользователя
                        item_scores[item] += similarity_score

            # Сортируем товары по убыванию "счета" (релевантности)
            sorted_items = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)

            # Извлекаем только ID товаров, отсекаем по top_k
            rec_items = [item_id for item_id, score in sorted_items[:top_k]]

            # Дополнительная проверка: если рекомендаций меньше top_k,
            # дополним популярными товарами (исключая уже купленные и уже добавленные)
            if len(rec_items) < top_k:
                rec_items_set = set(rec_items)
                for pop_item in popular_items:
                     if len(rec_items) >= top_k:
                        break
                     if pop_item not in bought_by_user and pop_item not in rec_items_set:
                         rec_items.append(pop_item)
                         rec_items_set.add(pop_item)

        else:
            # --- Сценарий 2: Нет истории или нет похожих ---
            # Используем базовый подход: популярные, исключая купленные
            rec_items = [item for item in popular_items if item not in bought_by_user][:top_k]

        # Сохраняем рекомендации для пользователя
        recommendations[uid] = rec_items

    print(f"Рекомендации сгенерированы для {len(recommendations):,} пользователей")
    return recommendations


In [30]:
recommendations_hybrid = generate_recommendations_hybrid(
    test_users=test_users_during_period,
    popular_items=popular_items,
    user_preferences=user_preferences_test,
    similar_users_dict=similar_users_dict,
    orders_df=orders_df,
    top_k=100,
    start_date='2025-07-02',  # Тот же период, что и для популярных товаров
    end_date='2025-07-15'     # Тот же период, что и для популярных товаров
)



=== 🎯 ГЕНЕРАЦИЯ ГИБРИДНЫХ РЕКОМЕНДАЦИЙ ===
Подготовка индекса покупок пользователей за последние 2 недели...
Индекс покупок создан для 217648 пользователей за период 2025-07-02 - 2025-07-15.


Генерация рекомендаций: 100%|██████████| 470347/470347 [00:18<00:00, 25049.28it/s]

Рекомендации сгенерированы для 470,347 пользователей


In [31]:
import os
from datetime import datetime
from tqdm import tqdm
import pandas as pd

# Указываем путь сохранения
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed'

def save_submission(recommendations_hybrid, filename_prefix='fr_ozon_baseline_analytic_submission'):
    # Генерируем временную метку
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"{filename_prefix}_{timestamp}.csv"
    full_path = os.path.join(SAVE_PATH, filename)

    print(f"🚀 Начинаем формирование submission-файла...")
    print(f"📂 Путь сохранения: {full_path}")

    rows = []
    for uid, items in tqdm(recommendations_hybrid.items(), desc='Формирование submission'):
        rows.append({
            'user_id': uid,
            'item_id_1 item_id_2 ... item_id_100': ' '.join(map(str, items))
        })

    df = pd.DataFrame(rows)
    df.to_csv(full_path, index=False)

    print(f"✅ Файл успешно создан!")
    print(f"📁 Сохранено как: {full_path}")
    print(f"📊 Количество записей: {len(df)}")

# Пример вызова
save_submission(recommendations_hybrid)

🚀 Начинаем формирование submission-файла...
📂 Путь сохранения: /content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed/fr_ozon_baseline_analytic_submission_20250903_102305.csv


Формирование submission: 100%|██████████| 470347/470347 [00:10<00:00, 45871.44it/s]


✅ Файл успешно создан!
📁 Сохранено как: /content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed/fr_ozon_baseline_analytic_submission_20250903_102305.csv
📊 Количество записей: 470347
